# Matching - PySpark version

In [ ]:
# matching.py

import os
import json
import numpy as np
import pandas as pd

from typing import List, Optional, Dict, Any

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from matching import run_summary_matching_pipeline, run_time_series_matching_pipeline, love_plot_from_spark

In [ ]:
%run ./matching

In [ ]:
# ============================================================
# Example execution
# ============================================================

# 依你的 Fabric Lakehouse 路徑調整
SOURCE_PATH = "/lakehouse/default/Files/month_data"

OUTPUT_SUMMARY = "Files/output/matching/summary_spark"
OUTPUT_TS = "Files/output/matching/time_series_spark"

LOVE_PLOT_SUMMARY = "Files/output/figures/love_plot_summary_spark.png"
LOVE_PLOT_TS = "Files/output/figures/love_plot_time_series_spark.png"


In [ ]:
# df_spark = spark.read.parquet("Files/output/data/monthly_agg.parquet")
print("Reading source parquet with Spark ...")
df_spark = spark.read.parquet(SOURCE_PATH)

month_result= pd.read_parquet("/lakehouse/default/Files/month_data")
print(month_result.shape)

# ============================================================
# Load data
# ============================================================
print("Start running")

df_spark = spark.read.parquet("output/data/monthly_agg.parquet")
df_all = df_spark.filter(F.col("price") == "all")

In [ ]:
# ============================================================
# Time series
# ============================================================
print("Running time series matching...")

res_ts = run_time_series_matching_pipeline(
    sdf=df_all,
    output_folder="output/matching/time_series",
    lookback_months=12,
    verbose=True
)

print(res_ts.keys())

# ============================================================
print("Computing balance tables...")

balance_ts = res_ts["balance"]
balance_ts.show(5)

# ============================================================
print("Generating plots...")

love_plot_from_spark(
    balance_ts,
    output_path="output/figures/love_plot_time_series.png",
    title="Time Series Matching"
)

In [ ]:
# ============================================================
# Summary 1
# ============================================================
print("Running summary matching 1...")

res_summary_1 = run_summary_matching_pipeline(
    sdf=df_all,
    output_folder="output/matching/summary_1",
    lookback_months=12,
    summary_vars=[
        "peak_mean",
        "mean_consumption",
        "peak_sd",
        "trend"
    ],
    verbose=True
)

print(res_summary_1.keys())

# ============================================================
print("Computing balance tables...")
balance_summary_1 = res_summary_1["balance"]
balance_summary_1.show(5)

# ============================================================
print("Generating plots...")
love_plot_from_spark(
    balance_summary_1,
    output_path="output/figures/love_plot_summary_1.png",
    title="Summary Matching 1"
)

In [ ]:
# ============================================================
# Summary 2
# ============================================================
print("Running summary matching 2...")

res_summary_2 = run_summary_matching_pipeline(
    sdf=df_all,
    output_folder="output/matching/summary_2",
    lookback_months=12,
    summary_vars=[
        "peak_mean",
        "mean_consumption",
        "variance_consumption",
        "trend"
    ],
    verbose=True
)

print(res_summary_2.keys())

# ============================================================
print("Computing balance tables...")
balance_summary_2 = res_summary_2["balance"]
balance_summary_2.show(5)

# ============================================================
print("Generating plots...")
love_plot_from_spark(
    balance_summary_2,
    output_path="output/figures/love_plot_summary_2.png",
    title="Summary Matching 2"
)

In [ ]:
# ============================================================
# Summary 3
# ============================================================
print("Running summary matching 3...")

res_summary_3 = run_summary_matching_pipeline(
    sdf=df_all,
    output_folder="output/matching/summary_3",
    lookback_months=12,
    summary_vars=[
        "peak_mean",
        "peak_sd",
        "trend"
    ],
    verbose=True
)

print(res_summary_3.keys())

# ============================================================
print("Computing balance tables...")
balance_summary_3 = res_summary_3["balance"]
balance_summary_3.show(5)

# ============================================================
print("Generating plots...")
love_plot_from_spark(
    balance_summary_3,
    output_path="output/figures/love_plot_summary_3.png",
    title="Summary Matching 3"
)

In [ ]:
# ============================================================
# Seasonal (High price period)
# ============================================================
print("Running seasonal summary matching...")

res_summary_season = run_summary_matching_pipeline(
    sdf=df_all,
    output_folder="output/matching/summary_season",
    lookback_months=24,
    match_months=[1, 2, 3, 11, 12],
    summary_vars=[
        "peak_mean",
        "peak_sd",
        "peak_volatility",
        "trend"
    ],
    verbose=True
)

print(res_summary_season.keys())

# ============================================================
print("Computing balance tables...")
balance_summary_season = res_summary_season["balance"]
balance_summary_season.show(5)

# ============================================================
print("Generating plots...")
love_plot_from_spark(
    balance_summary_season,
    output_path="output/figures/love_plot_summary_season.png",
    title="Summary Matching (High Price Period)"
)

In [ ]:
# ============================================================
# Calendar matching
# ============================================================
print("Running calendar matching...")

res_calendar = run_time_series_matching_pipeline(
    sdf=df_all,
    output_folder="output/matching/calendar",
    lookback_months=24,
    match_months=[1, 2, 3, 11, 12],
    verbose=True
)

print(res_calendar.keys())

# ============================================================
print("Computing balance tables...")
balance_calendar = res_calendar["balance"]
balance_calendar.show(5)

# ============================================================
print("Generating plots...")
love_plot_from_spark(
    balance_calendar,
    output_path="output/figures/love_plot_calendar.png",
    title="Calendar Matching"
)

In [ ]:
# ============================================================
# Save results (optional extra, if pipeline沒自動存)
# ============================================================
import os
import json

def save_matching(res, folder, config):
    os.makedirs(folder, exist_ok=True)

    res["matches"].write.mode("overwrite").parquet(f"{folder}/matches.parquet")
    res["profiles"].write.mode("overwrite").parquet(f"{folder}/profiles.parquet")
    res["balance"].write.mode("overwrite").parquet(f"{folder}/balance.parquet")

    with open(f"{folder}/config.json", "w") as f:
        json.dump(config, f, indent=2)


# Example (如果你想額外存 config)
save_matching(
    res_summary_1,
    "output/matching/summary_1",
    {
        "type": "summary",
        "lookback_months": 12,
        "vars": ["peak_mean", "mean_consumption", "peak_sd", "trend"]
    }
)